## 1.1

Dans un notebook, importez vos classes (Corpus, Document, SearchEngine) depuis vos modules, puis chargez votre corpus déjà sauvegardé (méthode load). Il ne faut pas réinterroger les API Reddit / Arxiv : on repart du fichier produit en Partie 1.

In [1]:
import pandas as pd

from search_engine import Corpus, Document, SearchEngine

corpus = Corpus("IA et machine learning")
corpus.load("data/corpus.demo.jsonl")
len(corpus)

15

## 1.2

Instanciez votre moteur : moteur = SearchEngine(corpus). Vérifiez rapidement, par deux ou trois requêtes en mots-clés, que la recherche fonctionne (rappel de la Partie 2).

In [2]:
moteur = SearchEngine(corpus)

for requete in ["neural networks", "machine learning", "termeintrouvablezzzz"]:
    print(requete)
    display(moteur.search(requete, 5))

neural networks


,id,titre,url,score
0,8,Towards Understanding the Impact of Plasticity...,http://arxiv.org/abs/2610.42100v1,0.031802
1,0,I built a free AI/ML quiz that explains every ...,https://www.reddit.com/r/learnmachinelearning/...,0.029478
2,7,Hyperbolic Inverse Reinforcement Learning,http://arxiv.org/abs/2610.42104v1,0.027392
3,2,Modeling Market Microstructure as a Dynamic Pa...,https://www.reddit.com/r/learnmachinelearning/...,0.023589
4,4,Modeling Market Microstructure as a Dynamic Pa...,https://www.reddit.com/r/MachineLearning/comme...,0.023589


machine learning


,id,titre,url,score
0,13,Generative Design and Graph Neural Network Scr...,http://arxiv.org/abs/2610.41932v1,0.061676
1,6,Hard Constraint Projection in a Physics Inform...,http://arxiv.org/abs/2610.42106v1,0.058364
2,0,I built a free AI/ML quiz that explains every ...,https://www.reddit.com/r/learnmachinelearning/...,0.036786
3,3,Did you miss getting on the AI hype train?,https://www.reddit.com/r/learnmachinelearning/...,0.029927
4,8,Towards Understanding the Impact of Plasticity...,http://arxiv.org/abs/2610.42100v1,0.006827


termeintrouvablezzzz


,id,titre,url,score


### Détails des documents

Afficher toutes les informations d’un document arXiv et d’une publication Reddit.

In [3]:
for origine in ("arxiv", "reddit"):
    doc_id, document = next(
        (doc_id, document)
        for doc_id, document in corpus.documents.items()
        if document.getType() == origine
    )
    print(f"Document {doc_id}")
    document.display()
    print()

Document 5
Title: Gradient Descent on Two ReLU Neurons: Global Landscape and Bifurcation Dynamics
Author: Binghua Li
Date: 2026-10-30T17:51:35+00:00
URL: http://arxiv.org/abs/2610.42118v1
Source: arxiv
Text: Understanding how gradient descent learns features remains a central challenge in neural network theory. We study this question in a minimal multi-index setting that already exhibits rich multi-phase gradient dynamics: a well-specified two-neuron ReLU teacher-student model under isotropic Gaussian inputs, trained by population gradient flow from small initialization. We show that the dynamics are organized by two directions: the "easy" bisector direction, which carries the leading signal, and the "hard" splitting direction, which governs specialization. To characterize the loss structure and learning behavior, we analyze the population squared-loss landscape and show that every critical point with both student weights nonzero is either a global minimum or a saddle. We then track th

### Recherche des occurrences

Afficher tous les passages contenant « neural ».

In [4]:
passages = corpus.search("neural")
print(f"Occurrences : {len(passages)}")
for passage in passages:
    print(passage)

Occurrences : 14
03 questions on classical ML, neural networks, LLMs, RAG, agents, 
 material stops at "what is a neural network". I kept getting aske
yTorch Geometric (PyG), Graph Neural Networks (or spatial-temporal
ularized) around that time. * Neural network started to work again
yTorch Geometric (PyG), Graph Neural Networks (or spatial-temporal
emains a central challenge in neural network theory. We study this
on matrix. A physics informed neural network (PINN) [Raissi et al.
ues including linear weights, neural networks, or other function a
tric organization observed in neural circuits, we introduce Hyperb
ss to a loss of plasticity in neural networks. However, plasticity
ary differential equations. A neural network learns a mapping from
We trained a convolutional neural network to detect ultra-short
e-field relaxation, and graph neural network (GNN) property predic
oposes B-ExposNet, a Bayesian neural network for spatial exposure 


### Concordancier

Afficher les occurrences de « neural » avec leur contexte gauche et droit.

In [5]:
display(corpus.concorde("neural", 30))

,contexte gauche,motif trouvé,contexte droit
0,"03 questions on classical ML,",neural,"networks, LLMs, RAG, agents,"
1,"material stops at ""what is a",neural,"network"". I kept getting aske"
2,"yTorch Geometric (PyG), Graph",Neural,Networks (or spatial-temporal
3,ularized) around that time. *,Neural,network started to work again
4,"yTorch Geometric (PyG), Graph",Neural,Networks (or spatial-temporal
5,emains a central challenge in,neural,network theory. We study this
6,on matrix. A physics informed,neural,network (PINN) [Raissi et al.
7,"ues including linear weights,",neural,"networks, or other function a"
8,tric organization observed in,neural,"circuits, we introduce Hyperb"
9,ss to a loss of plasticity in,neural,"networks. However, plasticity"


### Fréquences et matrices

Inspecter les fréquences des termes et un extrait des matrices TF et TF–IDF.

In [6]:
moteur.stats(10)
frequences = SearchEngine.get_word_frequencies(moteur.freq, 10)
display(frequences)

mots = frequences.index.tolist()
colonnes = [moteur.vocab[mot]["id"] for mot in mots]
for nom, matrice in (("TF", moteur.mat_TF), ("TF–IDF", moteur.mat_TFxIDF)):
    print(nom, matrice.shape)
    display(pd.DataFrame(
        matrice[:5, colonnes].toarray(),
        index=moteur.doc_ids[:5],
        columns=mots,
    ))

Distinct words: 1387
the     188
and     121
a       114
of       83
in       62
to       58
i        41
with     41
on       40
is       36


,tf,df
the,188,15
and,121,14
a,114,15
of,83,14
in,62,15
to,58,13
i,41,6
with,41,12
on,40,12
is,36,14


TF (15, 1387)


,the,and,a,of,in,to,i,with,on,is
0,8,9,3,1,4,0,5,3,4,1
1,5,4,5,0,1,0,3,0,0,3
2,14,17,11,5,2,9,14,4,7,2
3,7,14,10,10,8,4,3,3,0,2
4,14,17,11,5,2,9,14,4,7,2


TF–IDF (15, 1387)


,the,and,a,of,in,to,i,with,on,is
0,0.0,0.620936,0.0,0.068993,0.0,0.000000,4.581454,0.669431,0.892574,0.068993
1,0.0,0.275971,0.0,0.000000,0.0,0.000000,2.748872,0.000000,0.000000,0.206979
2,0.0,1.172879,0.0,0.344964,0.0,1.287908,12.828070,0.892574,1.562005,0.137986
3,0.0,0.965900,0.0,0.689929,0.0,0.572403,2.748872,0.669431,0.000000,0.137986
4,0.0,1.172879,0.0,0.344964,0.0,1.287908,12.828070,0.892574,1.562005,0.137986


### Sauvegarde et rechargement

Sauvegarder le corpus dans un fichier temporaire, puis le recharger.

In [7]:
from pathlib import Path
from tempfile import TemporaryDirectory

print("Avant sauvegarde :", repr(corpus))
with TemporaryDirectory() as repertoire:
    chemin = Path(repertoire) / "corpus.jsonl"
    corpus.save(chemin)
    print(chemin.read_text(encoding="utf-8").splitlines()[0])
    corpus.load(chemin)
print("Après rechargement :", repr(corpus))

Avant sauvegarde : Corpus(nom='neural networks', documents=15, authors=34)
{"record_type": "corpus", "nom": "neural networks", "id_document": 15}
Après rechargement : Corpus(nom='neural networks', documents=15, authors=34)


## 1.3

Sur un gros corpus, la construction de la matrice ou la boucle de recherche peut être longue. Ajoutez un compteur de progression à l’aide de la librairie tqdm (cf. https://tqdm.github.io) afin de visualiser l’avancement de la boucle.

In [8]:
for taille in (100, 1000, 10000):
    corpus.load(f"data/corpus.{taille}.jsonl")
    moteur = SearchEngine(corpus, show_progress=True)

Indexing:   0%|          | 0/100 [00:00<?, ?document/s]

Indexing: 100%|██████████| 100/100 [00:00<00:00, 106211.80document/s]

Indexing:   0%|          | 0/1000 [00:00<?, ?document/s]

Indexing: 100%|██████████| 1000/1000 [00:00<00:00, 79220.02document/s]

Indexing:   0%|          | 0/10000 [00:00<?, ?document/s]

Indexing:  47%|████▋     | 4682/10000 [00:00<00:00, 46806.76document/s]

Indexing:  94%|█████████▎| 9363/10000 [00:00<00:00, 40170.81document/s]

Indexing: 100%|██████████| 10000/10000 [00:00<00:00, 40779.52document/s]